# Listening Intention Shapes Music Streaming Behaviors — Analysis Notebook

Full reproducible analysis for the paper, followed by the six supplementary analyses.

**Structure**
- Section 0: Setup (install, imports, config, environment versions)
- Section 1: Data loading and cleaning
- Section 2: Sample characteristics
- Section 3: Hypothesis 1 — intent vs. skips/replays
- Section 4: Hypothesis 2 — age vs. skipping
- Section 5: Hypothesis 3 — intent vs. workplace outcomes
- Section 6: Exploratory — musical diversity, behavior, session length, gender
- Section 7: Multivariate models
- Section 8: Correlation overview
- Section 9: Supplementary analyses (1-6)

All analyses run off a single cleaned dataframe (`df`), so the notebook executes top to bottom.


## Section 0 — Setup

In [ ]:
# Install packages not pre-installed in Colab
!pip install scikit-posthocs networkx scikit-learn --quiet

In [ ]:
# All imports in one place
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from scipy import stats as sp_stats  # alias kept for readability in some cells

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.power import FTestAnovaPower
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.anova import anova_lm

import scikit_posthocs as sp
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Consistent, larger fonts for all figures (mentor asked for bigger figure text)
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Fixed seed so bootstrap CIs, k-means, and random forest are reproducible
np.random.seed(42)

In [ ]:
# Environment versions — record these so others can reproduce the results
!python --version
!pip freeze | grep -E "^(pandas|numpy|scipy|statsmodels|scikit-posthocs|scikit-learn|networkx|matplotlib|seaborn)=" 

## Section 1 — Data loading and cleaning

The raw Google Forms export is loaded, columns are renamed to short names, categorical
responses are normalized, ordinal buckets are converted to numeric midpoints, and the
missing `job_satisfaction` values are mean-imputed. An explicit `job_sat_imputed` flag
records which rows were imputed, so the supplementary complete-case check (Section 9.1)
does not rely on matching a magic number.

In [ ]:
# --- Load raw data ---
# Adjust this path to wherever the raw survey CSV lives.
from google.colab import drive
drive.mount('/content/drive')

RAW_PATH = '/content/drive/MyDrive/ColabMusicData/Music Survey 2.csv'
df = pd.read_csv(RAW_PATH)
print('Raw shape:', df.shape)

In [ ]:
# --- Rename long survey questions to short variable names ---
rename_dict = {
    'What age range are you in?': 'age',
    'What gender are you?': 'gender',
    'Do you live in the U.S.?': 'in_us',
    'How long do you typically listen to music in a single session?': 'session_length',
    'How would you describe your objective when listening to music during this session?': 'intent',
    'Approximately how many times did you skip a song during this session?': 'skips',
    'Approximately how many times did you replay a song/start it from the beginning during this session?': 'replays',
    'Approximately how many artists did you listen to during this session?': 'artist_count',
    'Approximately how many genres did you listen to during this session?': 'genre_count',
    'How would you describe your overall mood during this session?': 'mood',
    'If you typically listen to a session of music during your job, how would you describe your job satisfaction?': 'job_satisfaction',
    'If you typically listen to a session of music during your job, how would you describe your effectiveness in performing without mistakes?': 'job_effectiveness'
}
df = df.rename(columns=rename_dict)

In [ ]:
# --- Normalize categorical text responses ---
def normalize_gen(val):
    if not isinstance(val, str):
        return "Other"
    v = val.strip().lower()
    if v == "male":
        return "Male"
    elif v == "female":
        return "Female"
    else:  # other / prefer not to say
        return "Other"

def normalize_intent(val):
    if not isinstance(val, str):
        return "Other"
    if val == "Background (my main focus is something else, like working)":
        return "Background"
    elif val == "Intentional (my main focus is listening to the music)":
        return "Intentional"
    elif val == "A mix of both":
        return "Mix"
    else:
        return "Other"

df['gender'] = df['gender'].apply(normalize_gen)
df['intent'] = df['intent'].apply(normalize_intent)

In [ ]:
# --- Impute missing job_satisfaction, keeping an explicit flag of which rows were imputed ---
print('Missing job_satisfaction before imputation:', df['job_satisfaction'].isna().sum())

df['job_sat_imputed'] = df['job_satisfaction'].isna()          # True = this row was imputed
df['job_satisfaction'] = df['job_satisfaction'].fillna(df['job_satisfaction'].mean())

print('Imputed rows:', df['job_sat_imputed'].sum(),
      '| complete-case rows:', (~df['job_sat_imputed']).sum())

In [ ]:
# --- Convert ordinal buckets to numeric midpoints ---
age_mapping  = {'Below 18': 17, '18-29': 23.5, '30-49': 39.5, '50-69': 59.5, '70+': 75, 'Prefer not to say': 23.5}
time_mapping = {'Below 15 minutes': 10, '15-60 minutes': 37.5, '1-2 hours': 90, '2+ hours': 180}
play_mapping = {'0': 0, '1-5': 3, '6-10': 8, '11-20': 15.5, '20+': 25}
div_mapping  = {'1': 1, '2-4': 3, '5-10': 7.5, '10+': 12}

df['age']            = df['age'].map(age_mapping)
df['session_length'] = df['session_length'].map(time_mapping)
df['skips']          = df['skips'].map(play_mapping)
df['replays']        = df['replays'].map(play_mapping)
df['artist_count']   = df['artist_count'].map(div_mapping)
df['genre_count']    = df['genre_count'].map(div_mapping)

print('Cleaned shape:', df.shape)
df.head()

In [ ]:
# --- Save the cleaned dataset (used by the paper and shareable for reproduction) ---
df.to_csv('cleaned_music_data.csv', index=False)
print('Saved cleaned_music_data.csv')

## Section 2 — Sample characteristics

In [ ]:
# Distributions for demographics and key variables
print("Gender Distribution:\n", df['gender'].value_counts(normalize=True) * 100)
print("\nIntent Distribution:\n", df['intent'].value_counts(normalize=True) * 100)
print("\nAge Distribution:\n", df['age'].value_counts(normalize=True) * 100)
print("\nU.S. Distribution:\n", df['in_us'].value_counts(normalize=True) * 100)
print("\nSession Distribution:\n", df['session_length'].value_counts(normalize=True) * 100)

In [ ]:
# Table 1: mean outcomes by intent group
analysis_cols = ['session_length', 'skips', 'replays', 'artist_count', 'genre_count',
                 'mood', 'job_satisfaction', 'job_effectiveness']
summary_table = df.groupby('intent')[analysis_cols].mean()
print(summary_table.round(3))

basic_stats = df.groupby('intent')[['skips', 'replays', 'artist_count']].agg(['mean', 'median', 'std'])
print("\n", basic_stats.round(2))

In [ ]:
# Figure 2: demographic pie charts
demo_cols = ['age', 'gender', 'in_us']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['#4F81BD', '#C0504D', '#9BBB59', '#8064A2']

for i, col in enumerate(demo_cols):
    counts = df[col].value_counts()
    axes[i].pie(counts, labels=counts.index, autopct='%1.1f%%',
                startangle=140, colors=colors[:len(counts)])
    axes[i].set_title(f'Distribution by {col.replace("_", " ").title()}', pad=15)

plt.tight_layout()
plt.savefig('demographics_pie_charts.png', dpi=300)
plt.show()

## Section 3 — Hypothesis 1: intent vs. skips and replays

Kruskal-Wallis omnibus tests across the three intent groups. The correct post-hoc for a
Kruskal-Wallis test is **Dunn's test** (not Tukey HSD, which pairs with ANOVA), so post-hocs
below use Dunn's with Holm correction.

In [ ]:
print("Hypothesis 1: Intent vs. Skips/Replays\n")

for metric in ['skips', 'replays']:
    groups = [df[df['intent'] == i][metric].dropna() for i in ['Intentional', 'Background', 'Mix']]
    h_stat, p_val = stats.kruskal(*groups)

    n = sum(len(g) for g in groups)
    epsilon_sq = h_stat / ((n**2 - 1) / (n + 1)) if n > 1 else 0

    print(f"{metric.capitalize()} across Intent Groups:")
    print(f"    Kruskal-Wallis H: {h_stat:.3f}, p-value: {p_val:.4e}, epsilon^2: {epsilon_sq:.3f}")

    # Correct post-hoc for Kruskal-Wallis = Dunn's test (only meaningful if omnibus is significant)
    if p_val < 0.05:
        print(f"    Significant; Dunn's (Holm) post-hoc:")
        print(sp.posthoc_dunn(df[df['intent'].isin(['Intentional','Background','Mix'])],
                              val_col=metric, group_col='intent', p_adjust='holm').round(4))
    else:
        print("    No significant difference; no post-hoc performed.")
    print()

In [ ]:
# Post-hoc power analysis for the (non-significant) skips comparison
n_intentional = len(df[df['intent'] == 'Intentional']['skips'].dropna())
n_background  = len(df[df['intent'] == 'Background']['skips'].dropna())
n_mix         = len(df[df['intent'] == 'Mix']['skips'].dropna())
total_n = n_intentional + n_background + n_mix

print(f"Group sizes — Intentional: {n_intentional}, Background: {n_background}, Mix: {n_mix}")
print(f"Total N: {total_n}")

epsilon_sq = 0.014
f = np.sqrt(epsilon_sq / (1 - epsilon_sq))
print(f"\nCohen's f equivalent to observed effect: {f:.4f}")

power_analysis = FTestAnovaPower()
observed_power = power_analysis.solve_power(effect_size=f, nobs=total_n, alpha=0.05, k_groups=3)
print(f"Observed power for H1: {observed_power:.4f}")

min_detectable_f = power_analysis.solve_power(power=0.8, nobs=total_n, alpha=0.05, k_groups=3)
min_detectable_epsilon = min_detectable_f**2 / (1 + min_detectable_f**2)
print(f"Minimum detectable Cohen's f at 80% power: {min_detectable_f:.4f}")
print(f"Equivalent epsilon squared: {min_detectable_epsilon:.4f}")

In [ ]:
# Figure 4: violin plots of skips and replays by intent
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
subset = df[df['intent'].isin(['Intentional', 'Background', 'Mix'])]

sns.violinplot(data=subset, x='intent', y='skips', order=['Intentional', 'Mix', 'Background'],
               ax=axes[0], palette='Blues')
axes[0].set_title('Distribution of Skips by Listening Intent')
axes[0].set_xlabel('Listening Intent'); axes[0].set_ylabel('Skips')

sns.violinplot(data=subset, x='intent', y='replays', order=['Intentional', 'Mix', 'Background'],
               ax=axes[1], palette='Blues')
axes[1].set_title('Distribution of Replays by Listening Intent')
axes[1].set_xlabel('Listening Intent'); axes[1].set_ylabel('Replays')

plt.tight_layout()
plt.savefig('violin_skips_replays.png', dpi=150)
plt.show()

## Section 4 — Hypothesis 2: age vs. skipping

Spearman correlation with a bootstrap 95% confidence interval, plus the age-by-intent
stratified view.

In [ ]:
def spearman_ci(x, y, n_boot=5000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x); y = np.asarray(y); n = len(x)
    boot_rhos = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        r, _ = stats.spearmanr(x[idx], y[idx])
        boot_rhos.append(r)
    lower = np.percentile(boot_rhos, (100 - ci) / 2)
    upper = np.percentile(boot_rhos, 100 - (100 - ci) / 2)
    rho_full, p_full = stats.spearmanr(x, y)
    return rho_full, p_full, lower, upper

clean = df[['age', 'skips']].dropna()
rho, p, lo, hi = spearman_ci(clean['age'], clean['skips'])
print(f"Hypothesis 2 — Age vs Skips: rho={rho:.3f}, p={p:.4e}, 95% CI=[{lo:.3f}, {hi:.3f}]")
print("Supported." if (p < 0.05 and rho < 0) else "Not supported.")

In [ ]:
# Stratified: age-skip correlation within each intent group + age-by-intent pivot
print("Age correlations within each intent group:")
for intent_type in ['Background', 'Intentional', 'Mix']:
    group = df[df['intent'] == intent_type]
    rho_s, p_s = stats.spearmanr(group['age'], group['skips'], nan_policy='omit')
    rho_a, p_a = stats.spearmanr(group['age'], group['artist_count'], nan_policy='omit')
    print(f"  {intent_type:12s} Age vs Skips: rho={rho_s:.3f}, p={p_s:.4f} | "
          f"Age vs Artist Count: rho={rho_a:.3f}, p={p_a:.4f}")

print("\nMean skips by age and intent:")
subset = df[df['intent'].isin(['Background', 'Intentional', 'Mix'])]
print(subset.groupby(['age', 'intent'])['skips'].mean().unstack().round(2))

In [ ]:
# Figure 5: age vs skipping, stratified by intent
plt.figure(figsize=(10, 6))
sns.pointplot(data=subset, x='age', y='skips', hue='intent',
              markers=["o", "s", "^"], linestyles=["-", "--", "-."])
plt.title('Age vs. Skipping Behavior Split by Listening Intent')
plt.ylabel('Average Skips'); plt.xlabel('Age Group')
plt.tight_layout()
plt.savefig('age_skips_by_intent.png', dpi=150)
plt.show()

## Section 5 — Hypothesis 3: intent vs. workplace outcomes

In [ ]:
print("Hypothesis 3: Intent vs. Job Metrics (Background vs Intentional t-tests)\n")
for metric in ['job_effectiveness', 'job_satisfaction']:
    bg  = df[df['intent'] == 'Background'][metric].dropna()
    ic  = df[df['intent'] == 'Intentional'][metric].dropna()
    if len(bg) > 1 and len(ic) > 1:
        _, p_levene = stats.levene(bg, ic)
        t_stat, p_val = stats.ttest_ind(bg, ic, equal_var=(p_levene > 0.05))
        print(f"{metric.replace('_',' ').title()}: t={t_stat:.4f}, p={p_val:.4f}")

# Supplementary KW across all three intent groups for job effectiveness (cited in the paper)
groups = [df[df['intent'] == i]['job_effectiveness'].dropna() for i in ['Intentional', 'Background', 'Mix']]
H, p = stats.kruskal(*groups)
print(f"\nJob effectiveness across all 3 intent groups (KW): H={H:.3f}, p={p:.3f}")

## Section 6 — Exploratory analyses

### 6.1 Musical diversity and intent (Dunn's test, excluding "Other")

The residual "Other" category (~3.5%) is excluded so the omnibus and post-hoc tests use the
same three primary groups. This is the analysis behind the paper's artist-count result.

In [ ]:
# Restrict to the three primary intent groups
df_primary = df[df['intent'] != 'Other']

# Omnibus Kruskal-Wallis on artist_count
groups = [g['artist_count'].values for _, g in df_primary.groupby('intent')]
H, p = stats.kruskal(*groups)
print(f"Artist count across intent (KW, 3 groups): H={H:.3f}, p={p:.4f}")

# Post-hoc Dunn's test (Holm-corrected)
dunn = sp.posthoc_dunn(df_primary, val_col='artist_count', group_col='intent', p_adjust='holm')
print("\nDunn's (Holm) p-values:")
print(dunn.round(4))

In [ ]:
# Figure 6: mean artist count by intent
plt.figure(figsize=(7, 5))
ax = sns.barplot(data=df_primary, x='intent', y='artist_count',
                 order=['Intentional', 'Mix', 'Background'], capsize=0.1, color='steelblue')
ax.set_title('Average Artist Count by Listening Intent')
ax.set_xlabel('Listening Intent'); ax.set_ylabel('Mean Artist Count')
plt.tight_layout()
plt.savefig('artist_count_by_intent.png', dpi=150)
plt.show()

### 6.2 Behavioral metrics vs. musical diversity

In [ ]:
print("Correlations between behavior and diversity:")
for behavior in ['skips', 'replays']:
    for diversity in ['artist_count', 'genre_count']:
        rho, p_val = stats.spearmanr(df[behavior], df[diversity], nan_policy='omit')
        print(f"  {behavior} vs {diversity}: rho={rho:.3f}, p={p_val:.9f}")

# artist_count vs genre_count
rho, p_val = stats.spearmanr(df['artist_count'], df['genre_count'], nan_policy='omit')
print(f"\nArtist vs Genre count: rho={rho:.3f}, p={p_val:.4e}")

In [ ]:
# Figure 7: skips vs artist count
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x='skips', y='artist_count', color='blue', scatter_kws={'alpha':0.5})
plt.title('Relationship Between Skipping Behavior and Artist Diversity')
plt.xlabel('Number of Skips'); plt.ylabel('Unique Artist Count')
plt.tight_layout()
plt.savefig('skips_vs_artists.png', dpi=150)
plt.show()

### 6.3 Session length, gender, and age-effectiveness

In [ ]:
# Session length vs behavior
print("Session length vs behavior:")
for metric in ['skips', 'replays', 'genre_count']:
    rho, p_val = stats.spearmanr(df['session_length'], df[metric], nan_policy='omit')
    print(f"  session_length vs {metric}: rho={rho:.3f}, p={p_val:.4e}")

# Gender vs skips
male = df[df['gender'] == 'Male']['skips'].dropna()
female = df[df['gender'] == 'Female']['skips'].dropna()
t_stat, p_val = stats.ttest_ind(male, female)
print(f"\nGender vs skips: t={t_stat:.3f}, p={p_val:.4f}")

# Age vs job effectiveness
rho, p_val = stats.spearmanr(df['age'], df['job_effectiveness'], nan_policy='omit')
print(f"Age vs job effectiveness: rho={rho:.3f}, p={p_val:.4f}")

## Section 7 — Multivariate models

Ordinal logistic regression on skip category, a binary high-skipper logistic model with odds
ratios, and two Baron-Kenny mediation checks.

In [ ]:
# Ordinal logistic regression: skip category ~ intent + age + mood
def skips_to_ordinal(val):
    if val == 0: return 0
    elif val <= 3: return 1     # 1-5
    elif val <= 8: return 2     # 6-10
    elif val <= 15.5: return 3  # 11-20
    else: return 4              # 20+

df['skips_ord'] = df['skips'].apply(skips_to_ordinal)
df_model = df[df['intent'].isin(['Intentional', 'Background', 'Mix'])].copy()
df_model['intent_enc'] = df_model['intent'].map({'Intentional': 0, 'Background': 1, 'Mix': 2})
model_data = df_model[['skips_ord', 'intent_enc', 'age', 'mood']].dropna()

ord_mod = OrderedModel(model_data['skips_ord'],
                       model_data[['intent_enc', 'age', 'mood']], distr='logit')
ord_res = ord_mod.fit(method='bfgs', disp=False)
print(ord_res.summary())

In [ ]:
# Binary logistic regression: high-skipper ~ intent + age + mood, with odds ratios
median_skips = df['skips'].median()
df['high_skipper'] = (df['skips'] > median_skips).astype(int)
print(f"Median skips: {median_skips} | High skippers: {df['high_skipper'].sum()}, "
      f"Low skippers: {(df['high_skipper']==0).sum()}\n")

df_log = df[df['intent'].isin(['Intentional', 'Background', 'Mix'])].copy()
df_log['intent_enc'] = df_log['intent'].map({'Intentional': 0, 'Background': 1, 'Mix': 2})
log_data = df_log[['high_skipper', 'intent_enc', 'age', 'mood']].dropna()

logit_model = smf.logit('high_skipper ~ intent_enc + age + mood', data=log_data).fit(disp=False)
print(logit_model.summary())

or_table = np.exp(logit_model.conf_int(alpha=0.05))
or_table.columns = ['OR_lower', 'OR_upper']
or_table.insert(0, 'OR', np.exp(logit_model.params))
print("\nOdds ratios with 95% CI:")
print(or_table.round(3))

In [ ]:
# Mediation checks (Baron & Kenny steps)
med_data = df[['age', 'mood', 'job_effectiveness', 'skips']].dropna()

m1 = smf.ols('job_effectiveness ~ age', data=med_data).fit()
m2 = smf.ols('mood ~ age', data=med_data).fit()
print("Mediation A (age -> mood -> job_effectiveness):")
print(f"  age->job_eff: coef={m1.params['age']:.4f}, p={m1.pvalues['age']:.4f}")
print(f"  age->mood:    coef={m2.params['age']:.4f}, p={m2.pvalues['age']:.4f}")

div_data = df[['age', 'artist_count', 'skips']].dropna()
m4 = smf.ols('skips ~ age', data=div_data).fit()
m5 = smf.ols('artist_count ~ age', data=div_data).fit()
m6 = smf.ols('skips ~ age + artist_count', data=div_data).fit()
print("\nMediation B (age -> artist_count -> skips):")
print(f"  age->skips (total): coef={m4.params['age']:.4f}, p={m4.pvalues['age']:.4f}")
print(f"  age->artist_count:  coef={m5.params['age']:.4f}, p={m5.pvalues['age']:.4f}")
print(f"  age->skips (direct): coef={m6.params['age']:.4f}, p={m6.pvalues['age']:.4f}, "
      f"artist_count: coef={m6.params['artist_count']:.4f}, p={m6.pvalues['artist_count']:.4f}")

In [ ]:
# Figure 8: forest plot of odds ratios from the high-skipper logistic model
def forest_plot(model, title, filename, exponentiate=False):
    params = model.params
    conf = model.conf_int(); conf.columns = ['lower', 'upper']
    summary_df = pd.concat([params, conf], axis=1)
    summary_df.columns = ['coef', 'lower', 'upper']
    summary_df = summary_df.drop('Intercept', errors='ignore')
    if exponentiate:
        summary_df = np.exp(summary_df); xlabel = "Odds Ratio (95% CI)"; ref_line = 1
    else:
        xlabel = "Coefficient (95% CI)"; ref_line = 0
    fig, ax = plt.subplots(figsize=(7, 4))
    y_pos = np.arange(len(summary_df))
    ax.errorbar(summary_df['coef'], y_pos,
                xerr=[summary_df['coef'] - summary_df['lower'],
                      summary_df['upper'] - summary_df['coef']],
                fmt='o', color='black', ecolor='gray', capsize=4)
    ax.axvline(ref_line, linestyle='--', color='red', alpha=0.6)
    ax.set_yticks(y_pos); ax.set_yticklabels(summary_df.index)
    ax.set_xlabel(xlabel); ax.set_title(title)
    plt.tight_layout(); plt.savefig(filename, dpi=200); plt.show()

forest_plot(logit_model, "Logistic Regression: Predictors of High-Skipper Status",
            'forest_logistic.png', exponentiate=True)

## Section 8 — Correlation overview

In [ ]:
# Figure 9: correlation heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr(numeric_only=True)
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Discovery Map")
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Figure 10: correlation network (|r| >= 0.15)
numeric_cols = ['age', 'session_length', 'skips', 'replays', 'artist_count',
                'genre_count', 'mood', 'job_satisfaction', 'job_effectiveness']
corr_matrix = df[numeric_cols].corr(method='pearson')

G = nx.Graph(); G.add_nodes_from(numeric_cols)
THRESHOLD = 0.15
for i, c1 in enumerate(numeric_cols):
    for c2 in numeric_cols[i+1:]:
        r = corr_matrix.loc[c1, c2]
        if abs(r) >= THRESHOLD:
            G.add_edge(c1, c2, weight=r)

fig, ax = plt.subplots(figsize=(9, 7))
pos = nx.spring_layout(G, seed=42, k=0.8)
edges = G.edges(data=True)
weights = [d['weight'] for (_, _, d) in edges]
edge_colors = ['crimson' if w > 0 else 'steelblue' for w in weights]
widths = [abs(w) * 8 for w in weights]
nx.draw_networkx_nodes(G, pos, node_size=1800, node_color='lightgray', edgecolors='black', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=widths, alpha=0.7, ax=ax)
ax.set_title("Correlation Network (|r| >= 0.15) — red = positive, blue = negative")
ax.axis('off')
plt.tight_layout()
plt.savefig('correlation_network.png', dpi=200)
plt.show()

# Section 9 — Supplementary analyses

Six additional analyses (robustness checks, clustering, interactions, standardized skip rate,
nonlinearity, random forest). These run on the same cleaned `df` built in Section 1 — no
separate data file is needed.

### 9.1 Robustness check: job-satisfaction mean imputation

Re-run the key job-satisfaction analyses on the complete-case subsample (imputed rows
excluded, identified by the explicit `job_sat_imputed` flag from Section 1).

In [ ]:
df_cc = df[~df['job_sat_imputed']].copy()
print('Complete-case n =', len(df_cc), ' (full sample n =', len(df), ')')

g_bg_full = df[df['intent']=='Background']['job_satisfaction']
g_in_full = df[df['intent']=='Intentional']['job_satisfaction']
g_bg_cc   = df_cc[df_cc['intent']=='Background']['job_satisfaction']
g_in_cc   = df_cc[df_cc['intent']=='Intentional']['job_satisfaction']

t_full, p_full = stats.ttest_ind(g_bg_full, g_in_full)
t_cc, p_cc = stats.ttest_ind(g_bg_cc.dropna(), g_in_cc.dropna())
print(f'Imputed sample : t={t_full:.2f}, p={p_full:.3f}')
print(f'Complete case  : t={t_cc:.2f}, p={p_cc:.3f}')

# mood-job_satisfaction correlation, imputed vs complete-case
sub_full = df.dropna(subset=['mood', 'job_satisfaction'])
r_full, _ = stats.pearsonr(sub_full['mood'], sub_full['job_satisfaction'])
sub_cc = df_cc.dropna(subset=['mood', 'job_satisfaction'])
r_cc, _ = stats.pearsonr(sub_cc['mood'], sub_cc['job_satisfaction'])
print(f'\nmood~job_satisfaction  imputed r={r_full:.2f}  |  complete-case r={r_cc:.2f}')

In [ ]:
# Figure S1: mood-job_satisfaction correlation, imputed vs complete case
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharey=True)
for ax, data, title, col in [
    (axes[0], sub_full, f'Full sample (imputed), n={len(sub_full)}\nr={r_full:.2f}', '#9DB4CE'),
    (axes[1], sub_cc,   f'Complete case, n={len(sub_cc)}\nr={r_cc:.2f}', '#C0703C')]:
    jitter = np.random.uniform(-.15, .15, len(data))
    ax.scatter(data['mood'] + jitter, data['job_satisfaction'], alpha=.35, s=25, color=col)
    z = np.polyfit(data['mood'], data['job_satisfaction'], 1)
    xs = np.linspace(data['mood'].min(), data['mood'].max(), 50)
    ax.plot(xs, np.polyval(z, xs), color='#2A4D69', lw=2)
    ax.set_title(title); ax.set_xlabel('Mood')
axes[0].set_ylabel('Job Satisfaction')
plt.suptitle('Figure S1. Mean Imputation Attenuates the Mood-Job Satisfaction Association')
plt.tight_layout(); plt.savefig('figS1_imputation.png', dpi=150); plt.show()

### 9.2 Data-driven clustering vs. self-reported intent

In [ ]:
behav = df.dropna(subset=['skips','replays','artist_count','genre_count']).copy()
Xs = StandardScaler().fit_transform(behav[['skips','replays','artist_count','genre_count']])

print("Silhouette scores by k:")
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
    print(f"  k={k}  inertia={km.inertia_:.1f}  silhouette={silhouette_score(Xs, km.labels_):.3f}")

# k=3 for comparison with the paper's 3-group framework
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(Xs)
behav['cluster'] = km.labels_
prof = behav.groupby('cluster')[['skips','replays','artist_count','genre_count']].mean()
print("\nCluster profiles:\n", prof.round(2))
print("\nCluster sizes:\n", behav['cluster'].value_counts())

In [ ]:
# chi-square: behavioral cluster vs self-reported intent
main = behav[behav['intent'].isin(['Background','Intentional','Mix'])]
ct = pd.crosstab(main['cluster'], main['intent'])
chi2, pchi, dof, _ = stats.chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(ct)
print(f"\nChi2={chi2:.3f}, p={pchi:.4f}, dof={dof}, Cramer's V={cramers_v:.3f}")

# do clusters differ on demographics / outcomes?
print("\nCluster differences on other variables:")
for var in ['age','mood','job_effectiveness','job_satisfaction','session_length']:
    gps = [main[main['cluster']==c][var].dropna() for c in sorted(main['cluster'].unique())]
    H, p = stats.kruskal(*gps)
    print(f"  {var:18s} KW H={H:.3f}, p={p:.3f}")

In [ ]:
# Figure S3 + S4: cluster profiles and cluster composition by intent
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

metrics = ['skips','replays','artist_count','genre_count']
x = np.arange(len(metrics)); w = 0.25
for i, c in enumerate(sorted(behav['cluster'].unique())):
    axes[0].bar(x + (i-1)*w, prof.loc[c, metrics].values, w,
                label=f'Cluster {c} (n={sum(behav["cluster"]==c)})')
axes[0].set_xticks(x); axes[0].set_xticklabels(['Skips','Replays','Artist\nCount','Genre\nCount'])
axes[0].set_ylabel('Mean value'); axes[0].set_title('Figure S3. Behavioral Cluster Profiles (k=3)')
axes[0].legend(fontsize=9)

ct_pct = ct.div(ct.sum(axis=0), axis=1) * 100
bottom = np.zeros(ct_pct.shape[1]); colors_c = ['#4C72B0','#DD8452','#55A868']
for i, c in enumerate(ct_pct.index):
    axes[1].bar(ct_pct.columns, ct_pct.loc[c], bottom=bottom, label=f'Cluster {c}', color=colors_c[i])
    bottom += ct_pct.loc[c].values
axes[1].set_ylabel('% of self-reported intent group')
axes[1].set_title(f"Figure S4. Clusters vs Self-Reported Intent\n(chi2={chi2:.2f}, p={pchi:.2f}, V={cramers_v:.2f})")
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig('figS3_S4_clustering.png', dpi=150); plt.show()

### 9.3 Interaction effects (age x mood, gender x intent, residency x intent)

In [ ]:
# 9.3.1 Age x Mood on skips
d = df.dropna(subset=['skips','age','mood']).copy()
d['age_c'] = d['age'] - d['age'].mean()
d['mood_c'] = d['mood'] - d['mood'].mean()
m_int = smf.ols('skips ~ age_c * mood_c', data=d).fit()
print("Age x Mood interaction on skips:")
print(m_int.summary().tables[1])

# 9.3.2 Gender x Intent on skips
d2 = df[df['intent'].isin(['Background','Intentional','Mix']) & df['gender'].isin(['Male','Female'])].dropna(subset=['skips'])
aov = anova_lm(smf.ols('skips ~ C(gender) * C(intent)', data=d2).fit(), typ=2)
print("\nGender x Intent ANOVA:\n", aov)

# 9.3.3 US residency x Intent on skips
d3 = df[df['intent'].isin(['Background','Intentional','Mix']) & df['in_us'].isin(['Yes','No'])].dropna(subset=['skips'])
aov3 = anova_lm(smf.ols('skips ~ C(in_us) * C(intent)', data=d3).fit(), typ=2)
print("\nUS Residency x Intent ANOVA:\n", aov3)

In [ ]:
# Figure S5: age-skip slope across mood terciles
d['mood_group'] = pd.cut(d['mood'], bins=[0,6,8,10],
                         labels=['Low mood (<=6)','Mid mood (7-8)','High mood (9-10)'])
fig, ax = plt.subplots(figsize=(6.8, 4.5))
for i, mg in enumerate(['Low mood (<=6)','Mid mood (7-8)','High mood (9-10)']):
    sub = d[d['mood_group']==mg]
    if len(sub) < 5: continue
    z = np.polyfit(sub['age'], sub['skips'], 1)
    xs = np.linspace(sub['age'].min(), sub['age'].max(), 50)
    ax.scatter(sub['age'], sub['skips'], alpha=.25, s=18, color=f'C{i}')
    ax.plot(xs, np.polyval(z, xs), color=f'C{i}', lw=2.2, label=mg)
ax.set_xlabel('Age'); ax.set_ylabel('Skips')
ax.set_title('Figure S5. Age-Skip Slope Across Mood Levels')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('figS5_age_mood.png', dpi=150); plt.show()

### 9.4 Standardized skip rate (controlling for session length)

In [ ]:
df['skip_rate'] = df['skips'] / df['session_length'] * 60  # skips per hour

g_bg  = df[df['intent']=='Background']['skip_rate'].dropna()
g_in  = df[df['intent']=='Intentional']['skip_rate'].dropna()
g_mix = df[df['intent']=='Mix']['skip_rate'].dropna()
H, p = stats.kruskal(g_bg, g_in, g_mix)
print(f"skip_rate ~ intent (KW): H={H:.3f}, p={p:.3f}")
print(f"means -> Background={g_bg.mean():.2f}, Intentional={g_in.mean():.2f}, Mix={g_mix.mean():.2f}")

sub = df.dropna(subset=['age','skip_rate'])
rho, p2 = stats.spearmanr(sub['age'], sub['skip_rate'])
print(f"\nage ~ skip_rate (Spearman): rho={rho:.3f}, p={p2:.4f}, n={len(sub)}")

In [ ]:
# Figures S7 + S8: normalized skip rate by intent, and age vs skip rate
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(['Background','Intentional','Mix'],
            [g_bg.mean(), g_in.mean(), g_mix.mean()],
            yerr=[g_bg.sem(), g_in.sem(), g_mix.sem()], capsize=5,
            color=['#4C72B0','#DD8452','#55A868'])
axes[0].set_ylabel('Skips per hour of listening')
axes[0].set_title(f'Figure S7. Skip Rate by Intent\nKW H={H:.2f}, p={p:.3f}')

axes[1].scatter(sub['age'] + np.random.uniform(-1,1,len(sub)), sub['skip_rate'],
                alpha=.3, s=22, color='#4C72B0')
z = np.polyfit(sub['age'], sub['skip_rate'], 1)
xs = np.linspace(sub['age'].min(), sub['age'].max(), 50)
axes[1].plot(xs, np.polyval(z, xs), color='#2A4D69', lw=2.3)
axes[1].set_xlabel('Age'); axes[1].set_ylabel('Skips per hour')
axes[1].set_title(f'Figure S8. Age-Skip After Normalizing\n(rho={rho:.2f}, p={p2:.3f})')
plt.tight_layout(); plt.savefig('figS7_S8_skiprate.png', dpi=150); plt.show()

### 9.5 Nonlinearity check: is the age-skip relationship linear?

In [ ]:
d4 = df.dropna(subset=['skips','age']).copy()
d4['age_c'] = d4['age'] - d4['age'].mean()
m_lin  = smf.ols('skips ~ age_c', data=d4).fit()
m_quad = smf.ols('skips ~ age_c + I(age_c**2)', data=d4).fit()
print(anova_lm(m_lin, m_quad))
print("\nQuadratic term p-value:", m_quad.pvalues['I(age_c ** 2)'])

# Figure S9: linear vs quadratic fit
fig, ax = plt.subplots(figsize=(6.8, 4.5))
ax.scatter(d4['age'] + np.random.uniform(-1,1,len(d4)), d4['skips'], alpha=.25, s=20, color='gray')
xs = np.linspace(d4['age'].min(), d4['age'].max(), 100); xs_c = xs - d4['age'].mean()
ax.plot(xs, m_lin.params['Intercept'] + m_lin.params['age_c']*xs_c, color='#2A4D69', lw=2.3, label='Linear fit')
ax.plot(xs, m_quad.params['Intercept'] + m_quad.params['age_c']*xs_c + m_quad.params['I(age_c ** 2)']*xs_c**2,
        color='#C44E52', lw=2.3, ls='--', label='Quadratic fit')
ax.set_xlabel('Age'); ax.set_ylabel('Skips')
ax.set_title('Figure S9. Linear Model Is Adequate (quadratic p=%.2f)' % m_quad.pvalues['I(age_c ** 2)'])
ax.legend()
plt.tight_layout(); plt.savefig('figS9_nonlinearity.png', dpi=150); plt.show()

### 9.6 Random forest feature importance (non-parametric cross-check)

In [ ]:
dm = df[df['intent'].isin(['Background','Intentional','Mix'])].dropna(
    subset=['skips','age','mood','session_length','gender','in_us']).copy()
dm['intent_num'] = dm['intent'].map({'Background':0,'Intentional':1,'Mix':2})
dm['gender_num'] = dm['gender'].map({'Male':0,'Female':1,'Other':2})
dm['in_us_num']  = dm['in_us'].map({'Yes':1,'No':0,'Prefer not to say':np.nan})
dm = dm.dropna(subset=['in_us_num'])

features = ['age','mood','session_length','intent_num','gender_num','in_us_num']
X = dm[features]; y = dm['skips']

rf = RandomForestRegressor(n_estimators=500, random_state=42, max_depth=4, min_samples_leaf=5)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_,
                        index=['Age','Mood','Session Length','Intent','Gender','US Residency']
                        ).sort_values()
scores = cross_val_score(rf, X, y, cv=5, scoring='r2')
print("Feature importances:\n", importances.round(3))
print(f"\n5-fold CV R^2: mean={scores.mean():.3f} (low/negative = little out-of-sample predictive power)")

In [ ]:
# Figure S10: random forest feature importance
fig, ax = plt.subplots(figsize=(6.8, 4.5))
ax.barh(importances.index, importances.values, color='#55A868')
ax.set_xlabel('Random Forest Feature Importance')
ax.set_title('Figure S10. Mood and Age Rank Above Intent\nas Predictors of Skipping')
plt.tight_layout(); plt.savefig('figS10_rf.png', dpi=150); plt.show()

print("Note: CV R^2 is low, so read the importance RANKING as corroborating evidence for")
print('"age and mood matter more than intent," not as a strong standalone predictive model.')